# Module 02: Transfer Entropy, Delay Reconstruction & Network Inference

**Core Mathematical Framework:**
- **Schreiber (2000) Transfer Entropy:** $T_{Y \to X} = I(Y_{\text{past}}; X_{\text{future}} \mid X_{\text{past}})$
- **Information Dynamics Chain Rule:** $I(X_{\text{past}}, Y_{\text{past}}; X_t) = A_X + T_{Y \to X}$
- **Frenzel–Pompe / KSG k-NN Estimator:** $\hat{I}(X; Y \mid Z) = \psi(k) - \langle \psi(n_{xz}+1) + \psi(n_{yz}+1) - \psi(n_z+1) \rangle$
- **Statistical Rigor:** Circular time-shift surrogates ($p = \frac{1 + \#\{T_{\text{surr}} \ge T_{\text{emp}}\\}}{1 + N_{\text{surr}}}$) + Benjamini-Hochberg FDR.

In [ ]:
import subprocess
import sys
from pathlib import Path

def _ensure_toolkit_on_path():
    cwd = Path.cwd().resolve()
    for cand in (cwd, cwd.parent, cwd.parent.parent):
        if (cand / "core").is_dir():
            if str(cand) not in sys.path:
                sys.path.insert(0, str(cand))
            return cand
    clone_dir = cwd / "The-Collective-Intelligence-Toolkit"
    if not clone_dir.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/poria-azadi96/The-Collective-Intelligence-Toolkit.git", str(clone_dir)],
            check=True,
        )
    if str(clone_dir) not in sys.path:
        sys.path.insert(0, str(clone_dir))
    return clone_dir

REPO_ROOT = _ensure_toolkit_on_path()
import matplotlib.pyplot as plt
import ipywidgets as widgets
from core.transfer_entropy import (
    create_te_dashboard_figure,
    run_module02_tier1,
    run_module02_tier2,
    run_module02_tier3,
)
from core.real_world_demos import (
    demo_m02_brain_connectivity,
    demo_m02_finance_contagion,
    demo_m02_gene_regulatory_network,
)

def run_te_laboratory(
    motif_type='Common_Driver',
    coupling=0.70,
    n_samples=450,
    k_neighbors=5,
    n_surrogates=60,
):
    fig, stats = create_te_dashboard_figure(
        motif_type=motif_type,
        coupling=coupling,
        n_samples=n_samples,
        k_neighbors=k_neighbors,
        n_surrogates=n_surrogates,
    )
    plt.show()
    print(
        f"Motif: {motif_type} | Lag u={stats['best_lag']} | "
        f"Bivariate TE={stats['te_bivariate']:.4f} (p={stats['p_bivariate']:.3f}) | "
        f"Conditional TE={stats['te_conditional']:.4f} (p={stats['p_conditional']:.3f})"
    )

widgets.interact(
    run_te_laboratory,
    motif_type=widgets.Dropdown(
        options=['Common_Driver', 'Indirect_Cascade', 'Direct_Coupling', 'Unobserved_Confounder'],
        value='Common_Driver',
        description='Causal Motif:',
    ),
    coupling=widgets.FloatSlider(min=0.2, max=0.9, step=0.1, value=0.7, description='Coupling C:'),
    n_samples=widgets.IntSlider(min=250, max=800, step=50, value=450, description='Sample Size:'),
    k_neighbors=widgets.IntSlider(min=3, max=10, step=1, value=5, description='k-NN (k):'),
    n_surrogates=widgets.IntSlider(min=50, max=200, step=25, value=60, description='Surrogates:'),
);

## 3-Tier Experimental Protocol
- **Tier 1 (Observation):** Verify conditioning on $Z$ suppresses spurious bivariate links in `Common_Driver` and `Indirect_Cascade`.
- **Tier 2 (Delay Reconstruction):** Wibral et al. (2013) scan $T(u)$ over $u = 1..5$ on `Direct_Coupling` ($\tau=2$) + Chain-Rule decomposition.
- **Tier 3 (Confounder Stress Test):** Sweep sample size $250 \to 800$ on `Unobserved_Confounder` to expose the irreducible observational limit.

In [ ]:
def run_m02_tier_selector(tier='Tier 1: Confounder & Cascade Filtering'):
    if tier.startswith('Tier 1'):
        fig, res = run_module02_tier1()
    elif tier.startswith('Tier 2'):
        fig, res = run_module02_tier2()
    else:
        fig, res = run_module02_tier3()
    plt.show()
    print(res)

widgets.interact(
    run_m02_tier_selector,
    tier=widgets.Dropdown(
        options=[
            'Tier 1: Confounder & Cascade Filtering',
            'Tier 2: Wibral Delay Reconstruction (tau=2)',
            'Tier 3: Unobserved Confounder Sample Sweep',
        ],
        value='Tier 1: Confounder & Cascade Filtering',
        description='Protocol:',
    ),
);

## Module 02 Real-World Applications
1. **Computational Neuroscience:** 4-region effective brain connectivity (`THAL`, `PFC`, `PAR`, `MOT`).
2. **Quantitative Finance:** Multi-asset asymmetric volatility contagion & systemic risk hub discovery.
3. **Cellular & Systems Biology:** Hill-kinetics transcriptional cascade reconstruction.

In [ ]:
def run_m02_real_world_demo(application='1. Effective Brain Connectivity'):
    if application.startswith('1.'):
        fig, res = demo_m02_brain_connectivity()
        print('True Edges:', res['true_edges'], '| Recovered:', res['recovered_edges'])
    elif application.startswith('2.'):
        fig, res = demo_m02_finance_contagion()
        print('Systemic Out-Strength:', res['out_strength'])
    else:
        fig, res = demo_m02_gene_regulatory_network()
        print('True Edges:', res['true_edges'], '| Recovered:', res['recovered_edges'])
    plt.show()

widgets.interact(
    run_m02_real_world_demo,
    application=widgets.Dropdown(
        options=[
            '1. Effective Brain Connectivity',
            '2. Quantitative Finance Contagion',
            '3. Systems Biology Gene Regulatory Network',
        ],
        value='1. Effective Brain Connectivity',
        description='Case Study:',
    ),
);